# AlphaGen US RL Training on Colab

这个 notebook 会在 Colab 中完成以下流程：

1. 挂载 Google Drive。
2. 拉取 AlphaGen 仓库并安装依赖。
3. 用 Qlib 下载或复用缓存的美股日频数据。
4. 按 `train / valid / test` 三段时间切分数据。
5. 运行 AlphaGen 的强化学习训练。
6. 把训练产物和分段评估结果保存回 Google Drive。

默认使用 `SP500` 股票池；如果当前 Qlib 数据目录里没有这个股票池，notebook 会自动从已有股票池里挑选一个可用候选。

In [ ]:
# 作用：挂载 Google Drive，用于缓存 Qlib 数据、同步 checkpoint 和保存评估结果。
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

In [ ]:
# 作用：屏蔽 Colab/Jupyter 的已知废弃警告，减少训练日志噪声。
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"jupyter_client\.session",
)


In [ ]:
# 作用：集中定义仓库、数据、时间切分、训练超参和 Drive 同步的默认配置。
from pathlib import Path

REPO_URL = 'https://github.com/ZZZZkp/alphagen.git'
REPO_BRANCH = 'codex/upgrade-numpy2-runtime'

WORKDIR = Path('/content')
REPO_DIR = WORKDIR / 'alphagen'

DRIVE_ROOT = Path('/content/drive/MyDrive/alphagen_us_rl')
DRIVE_DATA_CACHE = DRIVE_ROOT / 'qlib_data' / 'us_data'
DRIVE_ARCHIVE_DIR = DRIVE_ROOT / 'qlib_archives'
DRIVE_DATA_ARCHIVE = DRIVE_ARCHIVE_DIR / 'us_data.tar.gz'
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs'
LOCAL_QLIB_DIR = WORKDIR / 'qlib_data' / 'us_data'

MAX_BACKTRACK_DAYS = 100
MAX_FUTURE_DAYS = 30
AUTO_TRAIN_START = '2010-01-01'
PRE_PANDEMIC_TEST_END = '2020-01-31'
VALID_TRADING_DAYS = 252
TEST_TRADING_DAYS = 252

SEGMENTS = None
CALENDAR_INFO = None

SEEDS = (0, 1, 2)
POOL_CAPACITY = 10
TRAINING_STEPS = 200_000
PPO_N_STEPS = 2048
BATCH_SIZE = 128
PRINT_EXPR = False
CHECKPOINT_EVERY_N_ROLLOUTS = 10
MODEL_CHECKPOINT_START_FRACTION = 0.5
CHECKPOINT_SELECTION_METRIC = 'valid_rank_icir'
DRIVE_SYNC_EVERY_N_LOCAL_CHECKPOINTS = 1
US_DELTA_TIMES = (1, 5, 10, 20, 40, 60)

PREFERRED_INSTRUMENTS = ('sp500', 'SP500', 'all', 'ALL')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_URL)
print('Drive root:', DRIVE_ROOT)
print('Drive archive cache:', DRIVE_DATA_ARCHIVE)
print(
    'Segments: auto-computed from calendars/day.txt '
    f'(train_start={AUTO_TRAIN_START}, split_end={PRE_PANDEMIC_TEST_END}, '
    f'valid_days={VALID_TRADING_DAYS}, '
    f'test_days={TEST_TRADING_DAYS}, backtrack={MAX_BACKTRACK_DAYS}, '
    f'future={MAX_FUTURE_DAYS})'
)
print('Seeds:', SEEDS)
print('US delta windows:', US_DELTA_TIMES)
print('Drive sync cadence (local checkpoints):', DRIVE_SYNC_EVERY_N_LOCAL_CHECKPOINTS)


In [ ]:
# 作用：拉取指定分支、安装缺失依赖，并确认当前 Colab 运行时的关键版本。
import os
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if REPO_DIR.exists():
    run(['git', 'fetch', '--all', '--tags'], cwd=str(REPO_DIR))
    run(['git', 'checkout', REPO_BRANCH], cwd=str(REPO_DIR))
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=str(REPO_DIR))
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])

from importlib.metadata import PackageNotFoundError, version as dist_version
from packaging.requirements import Requirement


def load_requirements(path: Path):
    requirements = []
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        requirements.append(Requirement(line))
    return requirements


def split_requirements(requirements):
    satisfied = []
    missing = []
    for req in requirements:
        try:
            current_version = dist_version(req.name)
        except PackageNotFoundError:
            missing.append(str(req))
            continue
        if req.specifier and not req.specifier.contains(current_version, prereleases=True):
            missing.append(str(req))
        else:
            satisfied.append((req.name, current_version))
    return satisfied, missing


run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
requirements = load_requirements(REPO_DIR / 'requirements.txt')
satisfied, missing = split_requirements(requirements)
print(f'Reusing {len(satisfied)} requirements from the current Colab runtime.')
if satisfied:
    print('Reused packages:', ', '.join(f'{name}=={version}' for name, version in satisfied))
if missing:
    print('Installing missing or outdated packages:', ', '.join(missing))
    run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade',
        *missing,
    ])
    print('Selected dependencies installed.')
else:
    print('All requirements are already satisfied by the current runtime.')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
import google.protobuf
import sklearn
import torch
from sb3_contrib.ppo_mask import MaskablePPO

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('protobuf:', google.protobuf.__version__)
print('scikit-learn:', sklearn.__version__)
print('Torch:', torch.__version__)
print('MaskablePPO import OK:', MaskablePPO.__name__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 作用：准备或复用 US Qlib 数据，并自动生成疫情前的 train/valid/test 划分。
import pandas as pd

from alphagen_qlib.cache import ensure_drive_qlib_data


def download_qlib_us_data(target_dir: Path):
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        run([
            sys.executable,
            '-m',
            'qlib.cli.data',
            'qlib_data',
            '--target_dir',
            str(target_dir),
            '--region',
            'us',
        ])
    except subprocess.CalledProcessError:
        from qlib.tests.data import GetData

        print('CLI 下载失败，尝试调用 qlib.tests.data.GetData().qlib_data(...)')
        GetData().qlib_data(target_dir=str(target_dir), region='us', exists_skip=True)


def build_segments_from_calendar(calendar_path: Path, split_end: str):
    calendar = pd.read_csv(calendar_path, header=None, names=['date'])
    calendar['date'] = pd.to_datetime(calendar['date'])

    usable_dates = pd.Index(
        calendar['date'].iloc[MAX_BACKTRACK_DAYS : len(calendar) - MAX_FUTURE_DAYS]
    )
    split_end_ts = pd.Timestamp(split_end)
    usable_dates = usable_dates[usable_dates <= split_end_ts]

    required_days = VALID_TRADING_DAYS + TEST_TRADING_DAYS + 1
    if len(usable_dates) < required_days:
        raise ValueError(
            'Qlib US calendar is too short for the requested pre-pandemic split strategy: '
            f'{len(usable_dates)} usable rows found before {split_end}, but at least {required_days} are required.'
        )

    train_start_idx = int(usable_dates.searchsorted(pd.Timestamp(AUTO_TRAIN_START)))
    if train_start_idx >= len(usable_dates):
        raise ValueError(
            f'AUTO_TRAIN_START={AUTO_TRAIN_START} is later than the last usable pre-pandemic date {usable_dates[-1].strftime("%Y-%m-%d")}. '
            'Please move AUTO_TRAIN_START earlier or relax PRE_PANDEMIC_TEST_END.'
        )
    last_train_end_idx = len(usable_dates) - VALID_TRADING_DAYS - TEST_TRADING_DAYS - 1
    if train_start_idx > last_train_end_idx:
        raise ValueError(
            'Not enough usable trading days to build pre-pandemic train/valid/test splits from the current '
            'Qlib US dataset. Please reduce VALID_TRADING_DAYS/TEST_TRADING_DAYS or relax '
            'PRE_PANDEMIC_TEST_END.'
        )

    valid_start_idx = last_train_end_idx + 1
    valid_end_idx = valid_start_idx + VALID_TRADING_DAYS - 1
    test_start_idx = valid_end_idx + 1

    def fmt(ts):
        return pd.Timestamp(ts).strftime('%Y-%m-%d')

    segments = {
        'train': (fmt(usable_dates[train_start_idx]), fmt(usable_dates[last_train_end_idx])),
        'valid': (fmt(usable_dates[valid_start_idx]), fmt(usable_dates[valid_end_idx])),
        'test': (fmt(usable_dates[test_start_idx]), fmt(usable_dates[-1])),
    }
    info = {
        'calendar_start': fmt(calendar['date'].iloc[0]),
        'calendar_end': fmt(calendar['date'].iloc[-1]),
        'usable_start': fmt(usable_dates[0]),
        'usable_end': fmt(usable_dates[-1]),
        'calendar_rows': int(len(calendar)),
        'usable_rows': int(len(usable_dates)),
        'requested_split_end': split_end,
        'selected_split_end': fmt(usable_dates[-1]),
    }
    return segments, info


try:
    cache_result = ensure_drive_qlib_data(
        local_dir=LOCAL_QLIB_DIR,
        drive_archive_path=DRIVE_DATA_ARCHIVE,
        legacy_drive_dir=DRIVE_DATA_CACHE,
        downloader=download_qlib_us_data,
    )
except FileNotFoundError as exc:
    raise FileNotFoundError(
        'Qlib US data preparation did not create calendars/day.txt. '
        '请检查当前 pyqlib 版本是否还能访问公开数据源，或改成你自己的 provider_uri。'
    ) from exc

print('Prepared US Qlib data from:', cache_result.source)
if cache_result.source == 'archive':
    print('Using cached US Qlib archive from Google Drive')
elif cache_result.source == 'legacy-directory':
    print('Using legacy Google Drive directory cache and packing it into an archive')
else:
    print('Downloaded US Qlib data and packed it into a Google Drive archive cache')

calendar_path = LOCAL_QLIB_DIR / 'calendars' / 'day.txt'
SEGMENTS, CALENDAR_INFO = build_segments_from_calendar(calendar_path, PRE_PANDEMIC_TEST_END)

instrument_files = sorted((LOCAL_QLIB_DIR / 'instruments').glob('*.txt'))
available_instruments = [path.stem for path in instrument_files]
print('Available instrument universes:', available_instruments[:20])

SELECTED_INSTRUMENT = next((name for name in PREFERRED_INSTRUMENTS if name in available_instruments), None)
if SELECTED_INSTRUMENT is None:
    raise ValueError(
        f'Could not find a supported US instrument universe in {available_instruments}. '
        '请把 PREFERRED_INSTRUMENTS 改成你的数据目录中真实存在的股票池名称。'
    )

instrument_path = LOCAL_QLIB_DIR / 'instruments' / f'{SELECTED_INSTRUMENT}.txt'
if instrument_path.exists():
    instrument_df = pd.read_csv(
        instrument_path,
        sep='	',
        header=None,
        names=['instrument', 'start', 'end'],
    )
    print(
        'Selected instrument coverage:',
        instrument_df['start'].min(),
        '->',
        instrument_df['end'].max(),
    )

print('Selected instrument universe:', SELECTED_INSTRUMENT)
print('Calendar coverage:', CALENDAR_INFO['calendar_start'], '->', CALENDAR_INFO['calendar_end'])
print('Usable AlphaGen range:', CALENDAR_INFO['usable_start'], '->', CALENDAR_INFO['usable_end'])
print('Auto-selected segments:', SEGMENTS)
print('Local Qlib dir:', LOCAL_QLIB_DIR)
print('Drive Qlib archive:', DRIVE_DATA_ARCHIVE)
print('Drive legacy cache dir:', DRIVE_DATA_CACHE)


In [ ]:
# 作用：把 RL 环境补丁成美股五特征版本，并同步更新动作空间和窗口设置。
import numpy as np
import alphagen.config as config_module
import alphagen.rl.env.wrapper as env_wrapper_module
import alphagen_qlib.stock_data as stock_data_module
import scripts.rl as rl_module
from alphagen.data.tokens import ConstantToken, DeltaTimeToken, ExpressionToken, FeatureToken, OperatorToken, SequenceIndicatorToken, SequenceIndicatorType
from alphagen_qlib.stock_data import FeatureType

US_FEATURES = (
    FeatureType.OPEN,
    FeatureType.CLOSE,
    FeatureType.HIGH,
    FeatureType.LOW,
    FeatureType.VOLUME,
)
US_FEATURE_NAMES = tuple(f.name.lower() for f in US_FEATURES)

config_module.DELTA_TIMES = list(US_DELTA_TIMES)
env_wrapper_module.DELTA_TIMES = list(US_DELTA_TIMES)

OriginalStockData = stock_data_module.StockData


class USStockData(OriginalStockData):
    def __init__(self, *args, features=None, **kwargs):
        if features is None:
            features = list(US_FEATURES)
        super().__init__(*args, features=features, **kwargs)


stock_data_module.StockData = USStockData
rl_module.StockData = USStockData


env_wrapper_module.SIZE_FEATURE = len(US_FEATURES)
env_wrapper_module.SIZE_DELTA_TIME = len(env_wrapper_module.DELTA_TIMES)
env_wrapper_module.SIZE_ACTION = (
    env_wrapper_module.SIZE_OP
    + env_wrapper_module.SIZE_FEATURE
    + env_wrapper_module.SIZE_DELTA_TIME
    + env_wrapper_module.SIZE_CONSTANT
    + env_wrapper_module.SIZE_SEP
)


def patched_action_masks(self):
    res = np.zeros(self.size_action, dtype=bool)
    valid = self.env.valid_action_types()

    offset = 0
    for i in range(offset, offset + env_wrapper_module.SIZE_OP):
        if valid['op'][env_wrapper_module.OPERATORS[i - offset].category_type()]:
            res[i] = True
    offset += env_wrapper_module.SIZE_OP
    if valid['select'][1]:
        res[offset:offset + env_wrapper_module.SIZE_FEATURE] = True
    offset += env_wrapper_module.SIZE_FEATURE
    if valid['select'][2]:
        res[offset:offset + env_wrapper_module.SIZE_CONSTANT] = True
    offset += env_wrapper_module.SIZE_CONSTANT
    if valid['select'][3]:
        res[offset:offset + env_wrapper_module.SIZE_DELTA_TIME] = True
    offset += env_wrapper_module.SIZE_DELTA_TIME
    if valid['select'][1]:
        res[offset:offset + len(self.subexprs)] = True
    offset += len(self.subexprs)
    if valid['select'][4]:
        res[offset] = True
    return res


def patched_action_to_token(self, action: int):
    if action < 0:
        raise ValueError
    if action < env_wrapper_module.SIZE_OP:
        return OperatorToken(env_wrapper_module.OPERATORS[action])
    action -= env_wrapper_module.SIZE_OP
    if action < env_wrapper_module.SIZE_FEATURE:
        return FeatureToken(US_FEATURES[action])
    action -= env_wrapper_module.SIZE_FEATURE
    if action < env_wrapper_module.SIZE_CONSTANT:
        return ConstantToken(env_wrapper_module.CONSTANTS[action])
    action -= env_wrapper_module.SIZE_CONSTANT
    if action < env_wrapper_module.SIZE_DELTA_TIME:
        return DeltaTimeToken(env_wrapper_module.DELTA_TIMES[action])
    action -= env_wrapper_module.SIZE_DELTA_TIME
    if action < len(self.subexprs):
        return ExpressionToken(self.subexprs[action])
    action -= len(self.subexprs)
    if action == 0:
        return SequenceIndicatorToken(SequenceIndicatorType.SEP)
    raise AssertionError('Invalid action index')


env_wrapper_module.AlphaEnvWrapper.action_masks = patched_action_masks
env_wrapper_module.AlphaEnvWrapper.action_to_token = patched_action_to_token

print('US notebook patch active: using features', US_FEATURE_NAMES)
print('RL feature action count:', env_wrapper_module.SIZE_FEATURE)
print('RL delta-time windows:', env_wrapper_module.DELTA_TIMES)


In [ ]:
# 作用：提供训练期 Drive 增量同步、从 Drive 续训和独立评测会复用的辅助函数。
import json
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from sb3_contrib.ppo_mask import MaskablePPO

from alphagen.data.expression import Feature, Ref
from alphagen.models.linear_alpha_pool import MseAlphaPool
from alphagen.rl.env.wrapper import AlphaEnv
from alphagen_qlib.calculator import QLibStockDataCalculator
from alphagen_qlib.stock_data import FeatureType, StockData, initialize_qlib
from alphagen_qlib.utils import load_alpha_pool_by_path


def build_target_expression():
    close = Feature(FeatureType.CLOSE)
    return Ref(close, -20) / close - 1


def resolve_drive_run_dir(run_name_or_path):
    path = Path(run_name_or_path)
    return path if path.is_absolute() else DRIVE_RUNS_DIR / path


def sync_run_directory_to_drive(local_run_dir):
    local_run_dir = Path(local_run_dir)
    drive_run_dir = DRIVE_RUNS_DIR / local_run_dir.name
    drive_run_dir.mkdir(parents=True, exist_ok=True)
    for path in local_run_dir.glob('*'):
        if path.is_file():
            shutil.copy2(path, drive_run_dir / path.name)
    return drive_run_dir


def build_split_calculators(segment_map, instrument, device):
    initialize_qlib(str(LOCAL_QLIB_DIR), region='us')
    target = build_target_expression()
    calculators = {}
    for split_name, (start_time, end_time) in segment_map.items():
        data = StockData(
            instrument=instrument,
            start_time=start_time,
            end_time=end_time,
            device=device,
        )
        calculators[split_name] = QLibStockDataCalculator(data, target)
    return calculators


def build_pool_for_calculator(calculator, pool_capacity, device, exprs=None, weights=None):
    pool = MseAlphaPool(
        capacity=pool_capacity,
        calculator=calculator,
        ic_lower_bound=None,
        l1_alpha=5e-3,
        device=device,
    )
    if exprs:
        pool.force_load_exprs(exprs, weights=weights)
    return pool


def evaluate_pool_metrics(pool_path, segment_map, instrument, pool_capacity, device):
    exprs, weights = load_alpha_pool_by_path(str(pool_path))
    calculators = build_split_calculators(segment_map, instrument, device)
    rows = []
    for split_name, (start_time, end_time) in segment_map.items():
        calculator = calculators[split_name]
        data = calculator.data
        pool = build_pool_for_calculator(
            calculator=calculator,
            pool_capacity=max(pool_capacity, len(exprs)),
            device=device,
            exprs=exprs,
            weights=weights,
        )
        ic, rank_ic = pool.test_ensemble(calculator)
        ic_mean, icir, rank_ic_mean, rank_icir = calculator.calc_pool_all_ret_with_ir(exprs, weights)
        rows.append(
            {
                'split': split_name,
                'start_time': start_time,
                'end_time': end_time,
                'n_days': int(data.n_days),
                'n_stocks': int(data.n_stocks),
                'ic': float(ic),
                'rank_ic': float(rank_ic),
                'icir': float(icir),
                'rank_icir': float(rank_icir),
                'pool_size': len(exprs),
                'checkpoint': Path(pool_path).name,
            }
        )
    return pd.DataFrame(rows)


def resolve_drive_pool_path(run_name_or_path, pool_filename=None):
    run_dir = resolve_drive_run_dir(run_name_or_path)
    if pool_filename is not None and pool_filename != '':
        pool_path = Path(pool_filename)
        return pool_path if pool_path.is_absolute() else run_dir / pool_path.name
    config_path = run_dir / 'colab_config.json'
    if config_path.exists():
        config = json.loads(config_path.read_text(encoding='utf-8'))
        best_checkpoint = config.get('best_checkpoint')
        if best_checkpoint:
            return run_dir / best_checkpoint
    pools = sorted(run_dir.glob('*_pool.json'), key=lambda path: int(path.name.split('_', 1)[0]))
    if not pools:
        raise FileNotFoundError(f'No *_pool.json found under {run_dir}')
    return pools[-1]


def resolve_checkpoint_base(run_name_or_path, checkpoint_name=None):
    run_dir = resolve_drive_run_dir(run_name_or_path)
    if checkpoint_name is None or checkpoint_name == '':
        pool_path = resolve_drive_pool_path(run_dir)
        checkpoint_name = pool_path.name
    name = Path(checkpoint_name).name
    if name.endswith('_pool.json'):
        stem = name[:-10]
    elif name.endswith('.zip'):
        stem = name[:-4]
    else:
        stem = name
    return run_dir / stem


def install_drive_sync_callback():
    if getattr(rl_module.CustomCallback, '_drive_sync_installed', False):
        return rl_module.CustomCallback

    BaseCallbackClass = rl_module.CustomCallback

    class DriveSyncCallback(BaseCallbackClass):
        _drive_sync_installed = True

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self._drive_local_checkpoint_count = 0

        def save_checkpoint(self, force=False):
            prev_pool_step = getattr(self, '_last_pool_checkpoint_step', None)
            prev_model_step = getattr(self, '_last_model_checkpoint_step', None)
            super().save_checkpoint(force=force)
            pool_changed = getattr(self, '_last_pool_checkpoint_step', None) != prev_pool_step
            model_changed = getattr(self, '_last_model_checkpoint_step', None) != prev_model_step
            if pool_changed or model_changed:
                self._drive_local_checkpoint_count += 1
                if force or self._drive_local_checkpoint_count % DRIVE_SYNC_EVERY_N_LOCAL_CHECKPOINTS == 0:
                    drive_run_dir = sync_run_directory_to_drive(self.save_path)
                    self._append_monitor_log(f'drive sync complete: {drive_run_dir}')

        def _on_training_end(self):
            result = super()._on_training_end()
            drive_run_dir = sync_run_directory_to_drive(self.save_path)
            self._append_monitor_log(f'final drive sync complete: {drive_run_dir}')
            return result

    rl_module.CustomCallback = DriveSyncCallback
    return DriveSyncCallback


install_drive_sync_callback()
print('Drive sync callback installed.')


In [ ]:
# 作用：按多个 seed 执行完整训练，并在训练过程中持续同步 checkpoint/pool 到 Drive。
import torch

from scripts.rl import run_single_experiment, status

if SEGMENTS is None:
    raise RuntimeError('SEGMENTS were not initialized. Please run the data-download cell first.')

install_drive_sync_callback()
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEGMENT_ORDER = ('train', 'valid', 'test')
SEGMENT_TUPLES = tuple(SEGMENTS[name] for name in SEGMENT_ORDER)
MODEL_CHECKPOINT_START_STEP = max(1, int(TRAINING_STEPS * MODEL_CHECKPOINT_START_FRACTION))
RUN_DIRS = []

print('Training device:', DEVICE)
print('Training segments:', dict(zip(SEGMENT_ORDER, SEGMENT_TUPLES)))
print('Checkpoint cadence (rollouts):', CHECKPOINT_EVERY_N_ROLLOUTS)
print('Model checkpoints start at step:', MODEL_CHECKPOINT_START_STEP)
print('Drive sync cadence (local checkpoints):', DRIVE_SYNC_EVERY_N_LOCAL_CHECKPOINTS)

for seed in SEEDS:
    run_dir = Path(
        run_single_experiment(
            seed=seed,
            instruments=SELECTED_INSTRUMENT,
            pool_capacity=POOL_CAPACITY,
            steps=TRAINING_STEPS,
            qlib_data_path=str(LOCAL_QLIB_DIR),
            qlib_region='us',
            device=DEVICE,
            segments=SEGMENT_TUPLES,
            ppo_n_steps=PPO_N_STEPS,
            batch_size=BATCH_SIZE,
            print_expr=PRINT_EXPR,
            checkpoint_every_n_rollouts=CHECKPOINT_EVERY_N_ROLLOUTS,
            model_checkpoint_start_step=MODEL_CHECKPOINT_START_STEP,
        )
    )
    RUN_DIRS.append(run_dir)
    drive_run_dir = sync_run_directory_to_drive(run_dir)
    print('Completed seed:', seed, '->', run_dir)
    print('Synced run directory to:', drive_run_dir)
    status(str(run_dir))

print('Run dirs:', RUN_DIRS)


In [ ]:
# 作用：从 Drive 中指定的 checkpoint 恢复模型和 alpha pool，继续追加训练。
import json
from pathlib import Path

import torch
from sb3_contrib.ppo_mask import MaskablePPO

from scripts.rl import status

DRIVE_RESUME_RUN = ''
DRIVE_RESUME_CHECKPOINT = ''
RESUME_EXTRA_STEPS = 50_000
RESUME_CHECKPOINT_EVERY_N_ROLLOUTS = CHECKPOINT_EVERY_N_ROLLOUTS
RESUME_MODEL_CHECKPOINT_START_STEP = 0

if DRIVE_RESUME_RUN == '':
    raise ValueError('请先设置 DRIVE_RESUME_RUN，例如某个 runs 目录名。')

install_drive_sync_callback()
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
source_run_dir = resolve_drive_run_dir(DRIVE_RESUME_RUN)
source_config = json.loads((source_run_dir / 'run_config.json').read_text(encoding='utf-8'))
source_segments = tuple(tuple(seg) for seg in source_config['segments'])
segment_names = ('train', 'valid', 'test')[:len(source_segments)]
segment_map = {name: seg for name, seg in zip(segment_names, source_segments)}
checkpoint_base = resolve_checkpoint_base(source_run_dir, DRIVE_RESUME_CHECKPOINT)
checkpoint_zip = checkpoint_base.with_suffix('.zip')
checkpoint_pool = Path(str(checkpoint_base) + '_pool.json')

if not checkpoint_zip.exists():
    raise FileNotFoundError(f'Checkpoint zip not found: {checkpoint_zip}')
if not checkpoint_pool.exists():
    raise FileNotFoundError(f'Checkpoint pool not found: {checkpoint_pool}')

calculators = build_split_calculators(segment_map, source_config['instruments'], DEVICE)
exprs, weights = load_alpha_pool_by_path(str(checkpoint_pool))
train_calculator = calculators[segment_names[0]]
pool = build_pool_for_calculator(
    calculator=train_calculator,
    pool_capacity=max(int(source_config['pool_capacity']), len(exprs)),
    device=DEVICE,
    exprs=exprs,
    weights=weights,
)
env = AlphaEnv(pool=pool, device=DEVICE, print_expr=bool(source_config.get('print_expr', False)))

resume_name = f"{source_run_dir.name}_resume_{datetime.now().strftime('%Y%m%d%H%M%S')}"
resume_run_dir = REPO_DIR / 'out' / 'results' / resume_name
resume_run_dir.mkdir(parents=True, exist_ok=True)
resume_config = dict(source_config)
resume_config.update(
    {
        'resumed_from_run': source_run_dir.name,
        'resumed_from_checkpoint': checkpoint_base.name,
        'resume_extra_steps': RESUME_EXTRA_STEPS,
        'checkpoint_every_n_rollouts': RESUME_CHECKPOINT_EVERY_N_ROLLOUTS,
        'model_checkpoint_start_step': RESUME_MODEL_CHECKPOINT_START_STEP,
    }
)
(resume_run_dir / 'run_config.json').write_text(json.dumps(resume_config, ensure_ascii=False, indent=2), encoding='utf-8')
(resume_run_dir / 'status.json').write_text(
    json.dumps(
        {
            'event': 'resume_created',
            'save_path': str(resume_run_dir),
            'num_timesteps': 0,
            'pool_size': int(pool.size),
            'pool_eval_cnt': int(pool.eval_cnt),
            'pool_best_ic_ret': float(pool.best_ic_ret),
            'significant_count': int((abs(pool.weights[:pool.size]) > 1e-4).sum()),
            'updated_at': datetime.now().isoformat(timespec='seconds'),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)

model = MaskablePPO.load(str(checkpoint_zip), env=env, device=DEVICE)
resume_callback = rl_module.CustomCallback(
    save_path=str(resume_run_dir),
    test_calculators=[calculators[name] for name in segment_names[1:]],
    verbose=1,
    llm_every_n_steps=int(source_config.get('llm_every_n_steps', 25_000)),
    drop_rl_n=int(source_config.get('drop_rl_n', 5)),
    checkpoint_every_n_rollouts=RESUME_CHECKPOINT_EVERY_N_ROLLOUTS,
    model_checkpoint_start_step=RESUME_MODEL_CHECKPOINT_START_STEP,
)
model.learn(
    total_timesteps=RESUME_EXTRA_STEPS,
    callback=resume_callback,
    tb_log_name=resume_name,
    reset_num_timesteps=False,
)

drive_resume_dir = sync_run_directory_to_drive(resume_run_dir)
print('Resumed run dir:', resume_run_dir)
print('Synced resumed artifacts to:', drive_resume_dir)
status(str(resume_run_dir))


In [ ]:
# 作用：对 Drive 中已有的 pool 做离线分段评测，并把结果写回对应 run 目录。
import json
from pathlib import Path

import torch

DRIVE_EVAL_RUN = ''
DRIVE_EVAL_POOL = ''

if DRIVE_EVAL_RUN == '':
    raise ValueError('请先设置 DRIVE_EVAL_RUN，例如某个 runs 目录名。')

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
drive_eval_dir = resolve_drive_run_dir(DRIVE_EVAL_RUN)
drive_eval_pool = resolve_drive_pool_path(drive_eval_dir, DRIVE_EVAL_POOL)
drive_eval_config = json.loads((drive_eval_dir / 'run_config.json').read_text(encoding='utf-8'))
drive_eval_segments = tuple(tuple(seg) for seg in drive_eval_config['segments'])
drive_eval_names = ('train', 'valid', 'test')[:len(drive_eval_segments)]
drive_eval_segment_map = {name: seg for name, seg in zip(drive_eval_names, drive_eval_segments)}

drive_eval_df = evaluate_pool_metrics(
    pool_path=drive_eval_pool,
    segment_map=drive_eval_segment_map,
    instrument=drive_eval_config['instruments'],
    pool_capacity=int(drive_eval_config['pool_capacity']),
    device=DEVICE,
)
drive_eval_df['run_dir'] = drive_eval_dir.name

eval_stem = Path(drive_eval_pool).name.replace('_pool.json', '')
eval_csv_path = drive_eval_dir / f'{eval_stem}_drive_eval.csv'
eval_json_path = drive_eval_dir / f'{eval_stem}_drive_eval.json'
drive_eval_df.to_csv(eval_csv_path, index=False)
eval_json_path.write_text(
    json.dumps(drive_eval_df.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Evaluated pool:', drive_eval_pool)
print(drive_eval_df)
print('Saved CSV to:', eval_csv_path)
print('Saved JSON to:', eval_json_path)


In [ ]:
# 作用：遍历每个 run 的全部 checkpoint，挑出最佳 checkpoint 并汇总多 seed 结果。
import json
import pandas as pd

from alphagen.data.expression import Feature, Ref
from alphagen.models.linear_alpha_pool import MseAlphaPool
from alphagen_qlib.calculator import QLibStockDataCalculator
from alphagen_qlib.stock_data import FeatureType, StockData, initialize_qlib
from alphagen_qlib.utils import load_alpha_pool_by_path

if not RUN_DIRS:
    raise RuntimeError('RUN_DIRS is empty. Please run the training cell first.')

initialize_qlib(str(LOCAL_QLIB_DIR), region='us')

close = Feature(FeatureType.CLOSE)
target = Ref(close, -20) / close - 1

split_calculators = {}
for split_name, (start_time, end_time) in SEGMENTS.items():
    data = StockData(
        instrument=SELECTED_INSTRUMENT,
        start_time=start_time,
        end_time=end_time,
        device=DEVICE,
    )
    split_calculators[split_name] = QLibStockDataCalculator(data, target)


def evaluate_checkpoint(pool_path):
    exprs, weights = load_alpha_pool_by_path(str(pool_path))
    rows = []
    for split_name, (start_time, end_time) in SEGMENTS.items():
        calculator = split_calculators[split_name]
        data = calculator.data
        pool = MseAlphaPool(
            capacity=max(POOL_CAPACITY, len(exprs)),
            calculator=calculator,
            ic_lower_bound=None,
            l1_alpha=5e-3,
            device=DEVICE,
        )
        pool.force_load_exprs(exprs, weights=weights)
        ic, rank_ic = pool.test_ensemble(calculator)
        ic_mean, icir, rank_ic_mean, rank_icir = calculator.calc_pool_all_ret_with_ir(exprs, weights)
        rows.append(
            {
                'split': split_name,
                'start_time': start_time,
                'end_time': end_time,
                'n_days': int(data.n_days),
                'n_stocks': int(data.n_stocks),
                'ic': float(ic),
                'rank_ic': float(rank_ic),
                'icir': float(icir),
                'rank_icir': float(rank_icir),
                'pool_size': len(exprs),
                'checkpoint': pool_path.name,
            }
        )
    return pd.DataFrame(rows)


seed_summaries = []
best_metrics_frames = []

for run_dir in RUN_DIRS:
    checkpoint_paths = sorted(
        run_dir.glob('*_steps_pool.json'),
        key=lambda path: int(path.name.split('_', 1)[0]),
    )
    if not checkpoint_paths:
        raise FileNotFoundError(f'No *_steps_pool.json checkpoint found under {run_dir}')

    run_config = json.loads((run_dir / 'run_config.json').read_text(encoding='utf-8'))
    seed = int(run_config['seed'])
    checkpoint_metrics = []
    checkpoint_summary_rows = []
    best_metrics_by_checkpoint = {}

    for pool_path in checkpoint_paths:
        metrics_df = evaluate_checkpoint(pool_path)
        metrics_df['seed'] = seed
        metrics_df['run_dir'] = run_dir.name
        checkpoint_metrics.append(metrics_df)
        best_metrics_by_checkpoint[pool_path.name] = metrics_df
        valid_row = metrics_df.loc[metrics_df['split'] == 'valid'].iloc[0]
        checkpoint_summary_rows.append(
            {
                'seed': seed,
                'run_dir': run_dir.name,
                'checkpoint': pool_path.name,
                'step': int(pool_path.name.split('_', 1)[0]),
                'valid_ic': float(valid_row['ic']),
                'valid_rank_ic': float(valid_row['rank_ic']),
                'valid_icir': float(valid_row['icir']),
                'valid_rank_icir': float(valid_row['rank_icir']),
            }
        )

    checkpoint_metrics_df = pd.concat(checkpoint_metrics, ignore_index=True)
    checkpoint_summary_df = pd.DataFrame(checkpoint_summary_rows).sort_values(
        by=['valid_rank_icir', 'valid_rank_ic', 'valid_ic'],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    best_checkpoint = checkpoint_summary_df.iloc[0]['checkpoint']
    best_metrics_df = best_metrics_by_checkpoint[best_checkpoint].copy()
    best_metrics_df['selected_by'] = CHECKPOINT_SELECTION_METRIC
    best_metrics_df['is_best_checkpoint'] = True

    checkpoint_metrics_path = run_dir / 'checkpoint_metrics.csv'
    checkpoint_selection_path = run_dir / 'checkpoint_selection.csv'
    best_metrics_path = run_dir / 'best_segment_metrics.csv'
    config_json_path = run_dir / 'colab_config.json'

    checkpoint_metrics_df.to_csv(checkpoint_metrics_path, index=False)
    checkpoint_summary_df.to_csv(checkpoint_selection_path, index=False)
    best_metrics_df.to_csv(best_metrics_path, index=False)
    config_json_path.write_text(
        json.dumps(
            {
                'repo_url': REPO_URL,
                'repo_branch': REPO_BRANCH,
                'seeds': list(SEEDS),
                'seed': seed,
                'pool_capacity': POOL_CAPACITY,
                'training_steps': TRAINING_STEPS,
                'ppo_n_steps': PPO_N_STEPS,
                'batch_size': BATCH_SIZE,
                'instrument': SELECTED_INSTRUMENT,
                'qlib_region': 'us',
                'qlib_data_path': str(LOCAL_QLIB_DIR),
                'segments': SEGMENTS,
                'calendar_info': CALENDAR_INFO,
                'feature_names': US_FEATURE_NAMES,
                'delta_times': list(US_DELTA_TIMES),
                'device': str(DEVICE),
                'checkpoint_every_n_rollouts': CHECKPOINT_EVERY_N_ROLLOUTS,
                'model_checkpoint_start_fraction': MODEL_CHECKPOINT_START_FRACTION,
                'model_checkpoint_start_step': MODEL_CHECKPOINT_START_STEP,
                'checkpoint_selection_metric': CHECKPOINT_SELECTION_METRIC,
                'drive_sync_every_n_local_checkpoints': DRIVE_SYNC_EVERY_N_LOCAL_CHECKPOINTS,
                'best_checkpoint': best_checkpoint,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )

    drive_run_dir = sync_run_directory_to_drive(run_dir)

    best_metrics_frames.append(best_metrics_df)
    checkpoint_summary = checkpoint_summary_df.iloc[0].to_dict()
    checkpoint_summary['best_checkpoint'] = best_checkpoint
    seed_summaries.append(checkpoint_summary)

    print(f'Seed {seed} best checkpoint: {best_checkpoint}')
    print(best_metrics_df)
    print('Synced run directory to:', drive_run_dir)

best_metrics_all_df = pd.concat(best_metrics_frames, ignore_index=True)
seed_summary_df = pd.DataFrame(seed_summaries)
aggregate_metrics_df = (
    best_metrics_all_df.groupby('split', as_index=False)
    .agg(
        seeds=('seed', 'nunique'),
        ic_mean=('ic', 'mean'),
        ic_std=('ic', 'std'),
        rank_ic_mean=('rank_ic', 'mean'),
        rank_ic_std=('rank_ic', 'std'),
        icir_mean=('icir', 'mean'),
        icir_std=('icir', 'std'),
        rank_icir_mean=('rank_icir', 'mean'),
        rank_icir_std=('rank_icir', 'std'),
        n_days=('n_days', 'first'),
        n_stocks_mean=('n_stocks', 'mean'),
    )
)

aggregate_dir = DRIVE_RUNS_DIR / 'aggregate'
aggregate_dir.mkdir(parents=True, exist_ok=True)
seed_summary_path = aggregate_dir / 'seed_summary.csv'
aggregate_metrics_path = aggregate_dir / 'best_checkpoint_metrics.csv'
best_metrics_all_path = aggregate_dir / 'best_segment_metrics_all.csv'
seed_summary_df.to_csv(seed_summary_path, index=False)
aggregate_metrics_df.to_csv(aggregate_metrics_path, index=False)
best_metrics_all_df.to_csv(best_metrics_all_path, index=False)

print('Per-seed checkpoint summary:')
print(seed_summary_df)
print('Multi-seed aggregate metrics (best checkpoint per seed):')
print(aggregate_metrics_df)
print('Saved aggregate seed summary to:', seed_summary_path)
print('Saved aggregate metrics to:', aggregate_metrics_path)
print('Saved best-per-seed segment metrics to:', best_metrics_all_path)


## 调参建议

- 当前 notebook 默认会把 `valid/test` 都限制在 `2020-01-31` 之前，避免把疫情冲击混进验证和测试。
- 本地 checkpoint 仍然按 `CHECKPOINT_EVERY_N_ROLLOUTS = 10` 节流生成，但现在会在训练过程中把新增 checkpoint/pool 增量同步到 Drive。
- 训练前半程默认只保存 `*_pool.json`，后半程才开始保存模型权重；如果你想更早保存模型，可以把 `MODEL_CHECKPOINT_START_FRACTION` 调小。
- 新增了两个工具 cell：一个从 Drive 里的 checkpoint 续训，一个对 Drive 里的 pool 做独立评测。
- 评估阶段会遍历每个 seed 的全部 `*_pool.json`，按 `valid_rank_icir` 选 best checkpoint，再汇总对应的 `test` 结果。
- 当前默认训练规模已经切到完整训练配置：`TRAINING_STEPS = 200_000`、`PPO_N_STEPS = 2048`，对应 `pool_capacity=10` 的完整训练档。
- notebook 现在会优先复用 `Google Drive/alphagen_us_rl/qlib_archives/us_data.tar.gz` 这个单文件归档缓存；如果只存在旧的目录缓存 `Google Drive/alphagen_us_rl/qlib_data/us_data`，会先复用它，再自动补写归档供后续运行加速。
- 如果公开 Qlib 数据下载通道暂时不可用，可以手动把你自己的 US Qlib 二进制数据打成 `us_data.tar.gz` 放到 `Google Drive/alphagen_us_rl/qlib_archives/`；旧目录缓存路径也仍然兼容。
